In [1]:
import gym
from gym import logger, spaces
from gym.envs.registration import register
from gym.envs.classic_control import CartPoleEnv

import GPy
import math
import numpy as np
from typing import Optional
from sklearn.metrics.pairwise import euclidean_distances
import matplotlib.pyplot as plt

import os
from functools import partial
from adv_RO import find_x_ro_action
from adv_RS1 import find_x_rs_action_RS1
from adv_RS2 import find_x_rs_action_RS2
from BO_loop import BO_loop_RS
from BO_loop import BO_loop_RO

In [ ]:
dir  = "util_arrays"

# Load the cost matrix
cost_matrix = np.load(f"{dir}/inverted_pend_4d_res8.npy")
print(cost_matrix.shape)

In [ ]:
from itertools import product

distance_metric = partial(euclidean_distances, squared=True)

# Load the cost matrix
dir  = "util_arrays"
cost_matrix = np.load(f"{dir}/inverted_pend_4d_res8.npy")

# Standardize the cost matrix
f_objective = -cost_matrix
f_objective = (f_objective - f_objective.mean()) / f_objective.std()
f_objective = np.clip(f_objective, -2, 2)

# Print function min, max and avg in one line
print(f"Min: {f_objective.min():.4f}, Max: {f_objective.max():.4f}, Avg: {f_objective.mean():.4f}")

# Define the range and resolution
theta_1_range = [-3, 2]
theta_2_range = [-3, 2]
theta_3_range = [-3, 2]
theta_u_range = [1, 5]
resolution = 8

# Create arrays of x and y values
theta_1 = np.linspace(theta_1_range[0], theta_1_range[1], resolution)
theta_u = np.linspace(theta_u_range[0], theta_u_range[1], resolution)

# Create a 4d meshgrid from theta arrays
product_list = [theta_1, theta_1, theta_1, theta_u]
# product_list = [theta_1, theta_u]
# product_list = [theta_1, theta_1, theta_u]
X = np.array(list(product(*product_list)))
Y = f_objective.flatten()

tau = 0.0
epsilon_t = 2.0
r_ro_true = epsilon_t * 1
r_ro_large = epsilon_t * 2.5
r_ro_small = epsilon_t * 0.3
attack_mode = 0

index_x_rs1, kappa = find_x_rs_action_RS1(X=X, Y=Y, tau=tau, dist_metric=euclidean_distances)
index_x_rs2, epsilon_sqr = find_x_rs_action_RS2(X=X, Y=Y, tau=tau, dist_metric=distance_metric)
index_x_ro_true = find_x_ro_action(X=X, Y=Y, r_ro=r_ro_true**2, dist_metric=distance_metric)
index_x_ro_large = find_x_ro_action(X=X, Y=Y, r_ro=r_ro_large**2, dist_metric=distance_metric)
index_x_ro_small = find_x_ro_action(X=X, Y=Y, r_ro=r_ro_small**2, dist_metric=distance_metric)

print(f"RS1: {X[index_x_rs1]}")
print(f"RS2: {X[index_x_rs2]}")
print(f"RO True: {X[index_x_ro_true]}")
print(f"RO Large: {X[index_x_ro_large]}")
print(f"RO Small: {X[index_x_ro_small]}")
# Print epsilon and kappa
print(f"RS2 epsilon: {np.sqrt(epsilon_sqr)}")
print("Percentage of points above tau:", np.mean(Y >= tau))

In [ ]:
kern = GPy.kern.RBF(input_dim=4, variance=0.4, lengthscale=[1.26, 1.9, 5.24, 1.86], ARD=True)

T=250
tau = 0.0
epsilon_t = 2.5
r_ro_true = epsilon_t * 1
r_ro_large = epsilon_t * 3
r_ro_small = epsilon_t * 0.3

distance_metric = partial(euclidean_distances, squared=True)
sigma_noise = 5e-2
num_initial_points = 5
num_iter = 100

path = "output_arrays"
os.makedirs(path, exist_ok=True)
attacks = [2]
Y = Y.reshape(-1, 1)
print(Y.shape)
        
import sys
sys.exit("Comement out to run the BO loops")

# Start the BO loops
for attack in attacks:
    attack_mode = attack

    # Name the arrays to save
    if np.isscalar(epsilon_t):
        array_RS1_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_tau_{tau}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RS1.npy'
        array_RS2_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_tau_{tau}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RS2.npy'
        array_RO_true_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RO.npy'
        array_RO_small_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RO.npy'
        array_RO_large_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t}_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RO.npy'
    elif isinstance(epsilon_t, np.ndarray):
        array_RS1_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_tau_{tau}_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RS1.npy'
        array_RS2_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_tau_{tau}_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RS2.npy'
        array_RO_true_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RO.npy'
        array_RO_small_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RO.npy'
        array_RO_large_name = f'inv_pend4d_res_{resolution}_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_arr_sigma_{sigma_noise:.0e}_T_{T}_attack_{attack_mode}_RO.npy'
    
    print(f'Attack mode: {attack_mode}')
    print(f'Num_iter: {num_iter}')

    print(f"RS1: {X[index_x_rs1]}")
    print('Starting RS1 loop')
    history_RS1 = BO_loop_RS(
        X=X, Y=Y, acquisition_function=find_x_rs_action_RS1, kernel=kern, sigma_noise=sigma_noise,
        num_init_samples=num_initial_points, distance_metric=euclidean_distances,
        attack_mode=attack_mode, tau=tau, epsilon_t=epsilon_t, T=T,
        print_samples=False, num_iter=num_iter, beta_t=5
    ) 
    # Save the history array of RS1
    np.save(os.path.join(path, array_RS1_name), history_RS1)

    print(f'RS2: {X[index_x_rs2]}')
    print('Starting RS2 loop')
    history_RS2 = BO_loop_RS(
        X=X, Y=Y, acquisition_function=find_x_rs_action_RS2, kernel=kern, sigma_noise=sigma_noise,
        num_init_samples=num_initial_points, distance_metric=distance_metric,
        attack_mode=attack_mode, tau=tau, epsilon_t=epsilon_t**2, T=T,
        print_samples=False, num_iter=num_iter, beta_t=5
    ) 
    # Save the history array of RS2
    np.save(os.path.join(path, array_RS2_name), history_RS2)
    
    print(f'RO True: {X[index_x_ro_true]}')
    print('Starting RO_true loop')
    history_RO_true = BO_loop_RO(
        X=X, Y=Y, num_init_samples=num_initial_points,
        kernel=kern, distance_metric=distance_metric, sigma_noise=sigma_noise,
        attack_mode=attack_mode, r_ro=epsilon_t**2, epsilon_t=epsilon_t**2, T=T, 
        print_samples=False, num_iter=num_iter, beta_t=5
    ) 
    # Save the history array of RO_true
    np.save(os.path.join(path, array_RO_true_name), history_RO_true)
    
    print(f'RO_small: {X[index_x_ro_small]}')
    print('Starting RO_small loop')
    history_RO_small = BO_loop_RO(
        X=X, Y=Y, num_init_samples=num_initial_points,
        sigma_noise=sigma_noise, kernel=kern, distance_metric=distance_metric,
        attack_mode=attack_mode, r_ro=r_ro_small**2, epsilon_t=epsilon_t**2, T=T, 
        print_samples=False, num_iter=num_iter, beta_t=5
    ) 
    # Save the history array of RO_small
    np.save(os.path.join(path, array_RO_small_name), history_RO_small)
    
    print(f'RO_large: {X[index_x_ro_large]}')
    print('Starting RO_large loop')
    history_RO_large = BO_loop_RO(
        X=X, Y=Y, num_init_samples=num_initial_points,
        sigma_noise=sigma_noise, kernel=kern, distance_metric=distance_metric,
        attack_mode=attack_mode, r_ro=r_ro_large**2, epsilon_t=epsilon_t**2, T=T,
        print_samples=False, num_iter=num_iter, beta_t=5
    ) 
    # Save the history array of RO_large
    np.save(os.path.join(path, array_RO_large_name), history_RO_large)


In [5]:
# num_iter = 50
# epsilon_t = 2.0
# r_ro_true = epsilon_t * 1
# r_ro_large = epsilon_t * 3
# r_ro_small = epsilon_t * 0.3

# folder_addition = "old/"
folder_addition = ""

# Load output arrays for plotting
data_RS1_0 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_tau_0.0_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_0_RS1.npy', allow_pickle=True)
data_RS2_0 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_tau_0.0_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_0_RS2.npy', allow_pickle=True)
data_RO_true_0 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_0_RO.npy', allow_pickle=True)
data_RO_small_0 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_0_RO.npy', allow_pickle=True)
data_RO_large_0 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_0_RO.npy', allow_pickle=True)

data_RS1_1 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_tau_0.0_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_1_RS1.npy', allow_pickle=True)
data_RS2_1 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_tau_0.0_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_1_RS2.npy', allow_pickle=True)
data_RO_true_1 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_1_RO.npy', allow_pickle=True)
data_RO_small_1 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_1_RO.npy', allow_pickle=True)
data_RO_large_1 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_1_RO.npy', allow_pickle=True)

data_RS1_2 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_tau_0.0_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_2_RS1.npy', allow_pickle=True)
data_RS2_2 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_tau_0.0_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_2_RS2.npy', allow_pickle=True)
data_RO_true_2 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_true:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_2_RO.npy', allow_pickle=True)
data_RO_small_2 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_small:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_2_RO.npy', allow_pickle=True)
data_RO_large_2 = np.load(f'output_arrays/{folder_addition}inv_pend4d_res_8_N_{num_iter}_r_{r_ro_large:.2f}_epsilon_{epsilon_t:.1f}_sigma_5e-02_T_250_attack_2_RO.npy', allow_pickle=True)

# data_RS1_0 = np.load('output_arrays/inv_pend4d_res_8_N_50_tau_0.0_epsilon_arr_sigma_5e-02_T_250_attack_0_RS1.npy', allow_pickle=True)
# data_RS2_0 = np.load('output_arrays/inv_pend4d_res_8_N_50_tau_0.0_epsilon_arr_sigma_5e-02_T_250_attack_0_RS2.npy', allow_pickle=True)
# data_RO_true_0 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_2.50_epsilon_arr_sigma_5e-02_T_250_attack_0_RO.npy', allow_pickle=True)
# data_RO_small_0 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_0.50_arr_sigma_5e-02_T_250_attack_0_RO.npy', allow_pickle=True)
# data_RO_large_0 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_5.00_arr_sigma_5e-02_T_250_attack_0_RO.npy', allow_pickle=True)

# data_RS1_1 = np.load('output_arrays/inv_pend4d_res_8_N_50_tau_0.0_epsilon_arr_sigma_5e-02_T_250_attack_1_RS1.npy', allow_pickle=True)
# data_RS2_1 = np.load('output_arrays/inv_pend4d_res_8_N_50_tau_0.0_epsilon_arr_sigma_5e-02_T_250_attack_1_RS2.npy', allow_pickle=True)
# data_RO_true_1 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_2.50_epsilon_arr_sigma_5e-02_T_250_attack_1_RO.npy', allow_pickle=True)
# data_RO_small_1 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_0.50_arr_sigma_5e-02_T_250_attack_1_RO.npy', allow_pickle=True)
# data_RO_large_1 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_5.00_arr_sigma_5e-02_T_250_attack_1_RO.npy', allow_pickle=True)

# data_RS1_2 = np.load('output_arrays/inv_pend4d_res_8_N_50_tau_0.0_epsilon_arr_sigma_5e-02_T_250_attack_2_RS1.npy', allow_pickle=True)
# data_RS2_2 = np.load('output_arrays/inv_pend4d_res_8_N_50_tau_0.0_epsilon_arr_sigma_5e-02_T_250_attack_2_RS2.npy', allow_pickle=True)
# data_RO_true_2 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_2.50_epsilon_arr_sigma_5e-02_T_250_attack_2_RO.npy', allow_pickle=True)
# data_RO_small_2 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_0.50_arr_sigma_5e-02_T_250_attack_2_RO.npy', allow_pickle=True)
# data_RO_large_2 = np.load('output_arrays/inv_pend4d_res_8_N_50_r_5.00_arr_sigma_5e-02_T_250_attack_2_RO.npy', allow_pickle=True)

In [ ]:
# Calculate Cumulative Regrets
temp, kappa_tau = find_x_rs_action_RS1(X=X, Y=Y, tau=tau, dist_metric=euclidean_distances)
print("tau:", tau)
print("epsilon_t:", epsilon_t)
print("Kappa_tau:", kappa_tau)

# Lenient regrets
cum_regret_RS1_0 = np.cumsum(np.maximum(tau - Y[data_RS1_0[:,1,:].astype(int)], 0), axis=1)
cum_regret_RS1_1 = np.cumsum(np.maximum(tau - Y[data_RS1_1[:,1,:].astype(int)], 0), axis=1)
cum_regret_RS1_2 = np.cumsum(np.maximum(tau - Y[data_RS1_2[:,1,:].astype(int)], 0), axis=1)
cum_regret_RS2_0 = np.cumsum(np.maximum(tau - Y[data_RS2_0[:,1,:].astype(int)], 0), axis=1)
cum_regret_RS2_1 = np.cumsum(np.maximum(tau - Y[data_RS2_1[:,1,:].astype(int)], 0), axis=1)
cum_regret_RS2_2 = np.cumsum(np.maximum(tau - Y[data_RS2_2[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_true_0 = np.cumsum(np.maximum(tau - Y[data_RO_true_0[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_true_1 = np.cumsum(np.maximum(tau - Y[data_RO_true_1[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_true_2 = np.cumsum(np.maximum(tau - Y[data_RO_true_2[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_small_0 = np.cumsum(np.maximum(tau - Y[data_RO_small_0[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_small_1 = np.cumsum(np.maximum(tau - Y[data_RO_small_1[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_small_2 = np.cumsum(np.maximum(tau - Y[data_RO_small_2[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_large_0 = np.cumsum(np.maximum(tau - Y[data_RO_large_0[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_large_1 = np.cumsum(np.maximum(tau - Y[data_RO_large_1[:,1,:].astype(int)], 0), axis=1)
cum_regret_RO_large_2 = np.cumsum(np.maximum(tau - Y[data_RO_large_2[:,1,:].astype(int)], 0), axis=1)

# RS regrets
cum_rs_regret_RS1_0 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS1_0[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RS1_1 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS1_1[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RS1_2 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS1_2[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RS2_0 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS2_0[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RS2_1 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS2_1[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RS2_2 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS2_2[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_true_0 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_true_0[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_true_1 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_true_1[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_true_2 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_true_2[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_small_0 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_small_0[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_small_1 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_small_1[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_small_2 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_small_2[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_large_0 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_large_0[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_large_1 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_large_1[:,1,:].astype(int)], 0), axis=1)
cum_rs_regret_RO_large_2 = np.cumsum(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_large_2[:,1,:].astype(int)], 0), axis=1)

# # Calculate Lenient Regrets
# avg_regret_RS1_0 = np.mean(np.maximum(tau - Y[data_RS1_0[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RS1_1 = np.mean(np.maximum(tau - Y[data_RS1_1[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RS1_2 = np.mean(np.maximum(tau - Y[data_RS1_2[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RS2_0 = np.mean(np.maximum(tau - Y[data_RS2_0[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RS2_1 = np.mean(np.maximum(tau - Y[data_RS2_1[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RS2_2 = np.mean(np.maximum(tau - Y[data_RS2_2[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_true_0 = np.mean(np.maximum(tau - Y[data_RO_true_0[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_true_1 = np.mean(np.maximum(tau - Y[data_RO_true_1[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_true_2 = np.mean(np.maximum(tau - Y[data_RO_true_2[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_small_0 = np.mean(np.maximum(tau - Y[data_RO_small_0[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_small_1 = np.mean(np.maximum(tau - Y[data_RO_small_1[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_small_2 = np.mean(np.maximum(tau - Y[data_RO_small_2[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_large_0 = np.mean(np.maximum(tau - Y[data_RO_large_0[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_large_1 = np.mean(np.maximum(tau - Y[data_RO_large_1[:,1,:].astype(int)], 0), axis=0)
# avg_regret_RO_large_2 = np.mean(np.maximum(tau - Y[data_RO_large_2[:,1,:].astype(int)], 0), axis=0)

# # Calculate RS Regrets
# temp, kappa_tau = find_x_rs_action_RS1(X=X, Y=Y, tau=tau, dist_metric=euclidean_distances)
# avg_rs_regret_RS1_0 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS1_0[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RS1_1 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS1_1[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RS1_2 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS1_2[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RS2_0 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS2_0[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RS2_1 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS2_1[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RS2_2 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RS2_2[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_true_0 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_true_0[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_true_1 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_true_1[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_true_2 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_true_2[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_small_0 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_small_0[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_small_1 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_small_1[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_small_2 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_small_2[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_large_0 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_large_0[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_large_1 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_large_1[:,1,:].astype(int)], 0), axis=0)
# avg_rs_regret_RO_large_2 = np.mean(np.maximum(tau - kappa_tau * epsilon_t - Y[data_RO_large_2[:,1,:].astype(int)], 0), axis=0)

# Calculate average cumulative regrets
avg_cum_reg_list_0 = [
    np.mean(cum_regret_RS1_0, axis=0),
    np.mean(cum_regret_RS2_0, axis=0),
    np.mean(cum_regret_RO_true_0, axis=0),
    np.mean(cum_regret_RO_small_0, axis=0),
    np.mean(cum_regret_RO_large_0, axis=0)
]
avg_cum_reg_list_1 = [
    np.mean(cum_regret_RS1_1, axis=0),
    np.mean(cum_regret_RS2_1, axis=0),
    np.mean(cum_regret_RO_true_1, axis=0),
    np.mean(cum_regret_RO_small_1, axis=0),
    np.mean(cum_regret_RO_large_1, axis=0)
]
avg_cum_reg_list_2 = [
    np.mean(cum_regret_RS1_2, axis=0),
    np.mean(cum_regret_RS2_2, axis=0),
    np.mean(cum_regret_RO_true_2, axis=0),
    np.mean(cum_regret_RO_small_2, axis=0),
    np.mean(cum_regret_RO_large_2, axis=0)
]
avg_cum_rs_reg_list_0 = [
    np.mean(cum_rs_regret_RS1_0, axis=0),
    np.mean(cum_rs_regret_RS2_0, axis=0),
    np.mean(cum_rs_regret_RO_true_0, axis=0),
    np.mean(cum_rs_regret_RO_small_0, axis=0),
    np.mean(cum_rs_regret_RO_large_0, axis=0)
]
avg_cum_rs_reg_list_1 = [
    np.mean(cum_rs_regret_RS1_1, axis=0),
    np.mean(cum_rs_regret_RS2_1, axis=0),
    np.mean(cum_rs_regret_RO_true_1, axis=0),
    np.mean(cum_rs_regret_RO_small_1, axis=0),
    np.mean(cum_rs_regret_RO_large_1, axis=0)
]
avg_cum_rs_reg_list_2 = [
    np.mean(cum_rs_regret_RS1_2, axis=0),
    np.mean(cum_rs_regret_RS2_2, axis=0),
    np.mean(cum_rs_regret_RO_true_2, axis=0),
    np.mean(cum_rs_regret_RO_small_2, axis=0),
    np.mean(cum_rs_regret_RO_large_2, axis=0)
]

# Calculate variance for error bars
var_cum_reg_list_0 = [
    np.std(cum_regret_RS1_0, axis=0),
    np.std(cum_regret_RS2_0, axis=0),
    np.std(cum_regret_RO_true_0, axis=0),
    np.std(cum_regret_RO_small_0, axis=0),
    np.std(cum_regret_RO_large_0, axis=0)
]
var_cum_reg_list_1 = [
    np.std(cum_regret_RS1_1, axis=0),
    np.std(cum_regret_RS2_1, axis=0),
    np.std(cum_regret_RO_true_1, axis=0),
    np.std(cum_regret_RO_small_1, axis=0),
    np.std(cum_regret_RO_large_1, axis=0)
]
var_cum_reg_list_2 = [
    np.std(cum_regret_RS1_2, axis=0),
    np.std(cum_regret_RS2_2, axis=0),
    np.std(cum_regret_RO_true_2, axis=0),
    np.std(cum_regret_RO_small_2, axis=0),
    np.std(cum_regret_RO_large_2, axis=0)
]
var_cum_rs_reg_list_0 = [
    np.std(cum_rs_regret_RS1_0, axis=0),
    np.std(cum_rs_regret_RS2_0, axis=0),
    np.std(cum_rs_regret_RO_true_0, axis=0),
    np.std(cum_rs_regret_RO_small_0, axis=0),
    np.std(cum_rs_regret_RO_large_0, axis=0)
]
var_cum_rs_reg_list_1 = [
    np.std(cum_rs_regret_RS1_1, axis=0),
    np.std(cum_rs_regret_RS2_1, axis=0),
    np.std(cum_rs_regret_RO_true_1, axis=0),
    np.std(cum_rs_regret_RO_small_1, axis=0),
    np.std(cum_rs_regret_RO_large_1, axis=0)
]
var_cum_rs_reg_list_2 = [
    np.std(cum_rs_regret_RS1_2, axis=0),
    np.std(cum_rs_regret_RS2_2, axis=0),
    np.std(cum_rs_regret_RO_true_2, axis=0),
    np.std(cum_rs_regret_RO_small_2, axis=0),
    np.std(cum_rs_regret_RO_large_2, axis=0)
]


# cum_reg_list_0 = [np.cumsum(avg_regret_RS1_0), np.cumsum(avg_regret_RS2_0), np.cumsum(avg_regret_RO_true_0),
#                 np.cumsum(avg_regret_RO_small_0), np.cumsum(avg_regret_RO_large_0)]
# cum_reg_list_1 = [np.cumsum(avg_regret_RS1_1), np.cumsum(avg_regret_RS2_1), np.cumsum(avg_regret_RO_true_1),
#                 np.cumsum(avg_regret_RO_small_1), np.cumsum(avg_regret_RO_large_1)]
# cum_reg_list_2 = [np.cumsum(avg_regret_RS1_2), np.cumsum(avg_regret_RS2_2), np.cumsum(avg_regret_RO_true_2),
#                 np.cumsum(avg_regret_RO_small_2), np.cumsum(avg_regret_RO_large_2)]

# cum_rs_reg_list_0 = [np.cumsum(avg_rs_regret_RS1_0), np.cumsum(avg_rs_regret_RS2_0), np.cumsum(avg_rs_regret_RO_true_0),
#                 np.cumsum(avg_rs_regret_RO_small_0), np.cumsum(avg_rs_regret_RO_large_0)]
# cum_rs_reg_list_1 = [np.cumsum(avg_rs_regret_RS1_1), np.cumsum(avg_rs_regret_RS2_1), np.cumsum(avg_rs_regret_RO_true_1),
#                 np.cumsum(avg_rs_regret_RO_small_1), np.cumsum(avg_rs_regret_RO_large_1)]
# cum_rs_reg_list_2 = [np.cumsum(avg_rs_regret_RS1_2), np.cumsum(avg_rs_regret_RS2_2), np.cumsum(avg_rs_regret_RO_true_2),
#                 np.cumsum(avg_rs_regret_RO_small_2), np.cumsum(avg_rs_regret_RO_large_2)]


In [ ]:
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})
# Ensure LaTeX is used for rendering text
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

fig, axs = plt.subplots(1, 3, figsize=(15, 3.5))
markevery = 49
alpha_error = 0.25

# Plotting cum_reg_list_0
axs[0].plot(avg_cum_reg_list_0[0], label='RS1', c='r', linestyle='--', marker='^', markevery=markevery)
axs[0].plot(avg_cum_reg_list_0[1], label='RS2', c='m', linestyle='--', marker='^', markevery=markevery)
axs[0].plot(avg_cum_reg_list_0[2], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery)
axs[0].plot(avg_cum_reg_list_0[3], label=r'RO $r=0.3 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery)
axs[0].plot(avg_cum_reg_list_0[4], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery)
# Plotting list_0 error bars at marker points
axs[0].errorbar(range(0, T, markevery), avg_cum_reg_list_0[0][::markevery].flatten(),
                yerr=var_cum_reg_list_0[0][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='r', 
)
axs[0].errorbar(range(0, T, markevery), avg_cum_reg_list_0[1][::markevery].flatten(),
                yerr=var_cum_reg_list_0[1][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='m', 
)
axs[0].errorbar(range(0, T, markevery), avg_cum_reg_list_0[2][::markevery].flatten(),
                yerr=var_cum_reg_list_0[2][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='b', 
)
axs[0].errorbar(range(0, T, markevery), avg_cum_reg_list_0[3][::markevery].flatten(),
                yerr=var_cum_reg_list_0[3][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='g', 
)
axs[0].errorbar(range(0, T, markevery), avg_cum_reg_list_0[4][::markevery].flatten(),
                yerr=var_cum_reg_list_0[4][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='darkred', 
)
# Labeling the axes
axs[0].set_xlabel('Round t')
axs[0].set_ylabel('Lenient Regret')
axs[0].set_title('Random Attack')

# Plotting avg_cum_reg_list_1
axs[1].plot(avg_cum_reg_list_1[0], label='RS1', c='r', linestyle='--', marker='^', markevery=markevery)
axs[1].plot(avg_cum_reg_list_1[1], label='RS2', c='m', linestyle='--', marker='^', markevery=markevery)
axs[1].plot(avg_cum_reg_list_1[2], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery)
axs[1].plot(avg_cum_reg_list_1[3], label=r'RO $r=0.3 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery)
axs[1].plot(avg_cum_reg_list_1[4], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery)
# Plotting list_1 error bars at marker points
axs[1].errorbar(range(0, T, markevery), avg_cum_reg_list_1[0][::markevery].flatten(),
                yerr=var_cum_reg_list_1[0][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='r', 
)
axs[1].errorbar(range(0, T, markevery), avg_cum_reg_list_1[1][::markevery].flatten(),
                yerr=var_cum_reg_list_1[1][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='m', 
)
axs[1].errorbar(range(0, T, markevery), avg_cum_reg_list_1[2][::markevery].flatten(),
                yerr=var_cum_reg_list_1[2][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='b', 
)
axs[1].errorbar(range(0, T, markevery), avg_cum_reg_list_1[3][::markevery].flatten(),
                yerr=var_cum_reg_list_1[3][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='g', 
)
axs[1].errorbar(range(0, T, markevery), avg_cum_reg_list_1[4][::markevery].flatten(),
                yerr=var_cum_reg_list_1[4][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='darkred', 
)
# Labeling the axes
axs[1].set_xlabel('Round t')
axs[1].set_ylabel('Lenient Regret')
axs[1].set_title('LCB Attack')
axs[1].set_ylim([-1, 100])  # Adjust the y-axis limits as needed

# Plotting avg_cum_reg_list_2
axs[2].plot(avg_cum_reg_list_2[0], label='RS1', c='r', linestyle='--', marker='^', markevery=markevery)
axs[2].plot(avg_cum_reg_list_2[1], label='RS2', c='m', linestyle='--', marker='^', markevery=markevery)
axs[2].plot(avg_cum_reg_list_2[2], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery)
axs[2].plot(avg_cum_reg_list_2[3], label=r'RO $r=0.3 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery)
axs[2].plot(avg_cum_reg_list_2[4], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery)
# Plotting list_2 error bars at marker points
axs[2].errorbar(range(0, T, markevery), avg_cum_reg_list_2[0][::markevery].flatten(),
                yerr=var_cum_reg_list_2[0][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='r', 
)
axs[2].errorbar(range(0, T, markevery), avg_cum_reg_list_2[1][::markevery].flatten(),
                yerr=var_cum_reg_list_2[1][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='m', 
)
axs[2].errorbar(range(0, T, markevery), avg_cum_reg_list_2[2][::markevery].flatten(),
                yerr=var_cum_reg_list_2[2][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='b', 
)
axs[2].errorbar(range(0, T, markevery), avg_cum_reg_list_2[3][::markevery].flatten(),
                yerr=var_cum_reg_list_2[3][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='g', 
)
axs[2].errorbar(range(0, T, markevery), avg_cum_reg_list_2[4][::markevery].flatten(),
                yerr=var_cum_reg_list_2[4][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='darkred', 
)
# Labeling the axes
axs[2].set_xlabel('Round t')
axs[2].set_ylabel('Lenient Regret')
axs[2].set_title('Worst Case Attack')
axs[2].set_ylim([-1, 100])  # Adjust the y-axis limits as needed

# Adding legends for tau and epsilon_t
axs[0].legend(title=r'$\tau=0$, $\epsilon_t=2.5$')
# axs[1].legend(title=r'$\tau=0$, $\epsilon_t=2.0$')
# axs[2].legend(title=r'$\tau=0$, $\epsilon_t=2.0$')

plt.tight_layout()
# Save as pdf
dir = 'figures'
os.makedirs(dir, exist_ok=True)
# plt.savefig(os.path.join(dir, 'regret_inv_pend_res8_N_100.pdf'), format='pdf', bbox_inches='tight')
# Display the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})
# Ensure LaTeX is used for rendering text
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

fig, axs = plt.subplots(1, 3, figsize=(15, 3.5))
markevery = 49
alpha_error = 0.25

# Plotting avg_cum_rs_reg_list_0
axs[0].plot(avg_cum_rs_reg_list_0[0], label='RS1', c='r', linestyle='--', marker='^', markevery=markevery)
axs[0].plot(avg_cum_rs_reg_list_0[1], label='RS2', c='m', linestyle='--', marker='^', markevery=markevery)
axs[0].plot(avg_cum_rs_reg_list_0[2], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery)
axs[0].plot(avg_cum_rs_reg_list_0[3], label=r'RO $r=0.3 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery)
axs[0].plot(avg_cum_rs_reg_list_0[4], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery)
# Plotting list_0 error bars at marker points
axs[0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_0[0][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_0[0][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='r', 
)
axs[0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_0[1][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_0[1][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='m', 
)
axs[0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_0[2][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_0[2][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='b', 
)
axs[0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_0[3][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_0[3][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='g', 
)
axs[0].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_0[4][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_0[4][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='darkred', 
)
# Labeling the axes
axs[0].set_xlabel('Round t')
axs[0].set_ylabel('Cumulative RS Regret')
axs[0].set_title('Random Attack')

# Plotting avg_cum_rs_reg_list_1
axs[1].plot(avg_cum_rs_reg_list_1[0], label='RS1', c='r', linestyle='--', marker='^', markevery=markevery)
axs[1].plot(avg_cum_rs_reg_list_1[1], label='RS2', c='m', linestyle='--', marker='^', markevery=markevery)
axs[1].plot(avg_cum_rs_reg_list_1[2], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery)
axs[1].plot(avg_cum_rs_reg_list_1[3], label=r'RO $r=0.3 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery)
axs[1].plot(avg_cum_rs_reg_list_1[4], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery)
# Plotting list_1 error bars at marker points
axs[1].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_1[0][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_1[0][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='r', 
)
axs[1].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_1[1][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_1[1][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='m', 
)
axs[1].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_1[2][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_1[2][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='b', 
)
axs[1].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_1[3][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_1[3][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='g', 
)
axs[1].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_1[4][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_1[4][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='darkred', 
)
# Labeling the axes
axs[1].set_xlabel('Round t')
axs[1].set_ylabel('Cumulative RS Regret')
axs[1].set_title('LCB Attack')
# axs[1].set_ylim([0, 20])  # Adjust the y-axis limits as needed

# Plotting avg_cum_rs_reg_list_2
axs[2].plot(avg_cum_rs_reg_list_2[0], label='RS1', c='r', linestyle='--', marker='^', markevery=markevery)
axs[2].plot(avg_cum_rs_reg_list_2[1], label='RS2', c='m', linestyle='--', marker='^', markevery=markevery)
axs[2].plot(avg_cum_rs_reg_list_2[2], label=r'RO $r=\epsilon_t$', c='b', linestyle='--', marker='d', markevery=markevery)
axs[2].plot(avg_cum_rs_reg_list_2[3], label=r'RO $r=0.3 \cdot \epsilon_t$', c='g', linestyle='--', marker='d', markevery=markevery)
axs[2].plot(avg_cum_rs_reg_list_2[4], label=r'RO $r=3 \cdot \epsilon_t$', c='darkred', linestyle='--', marker='d', markevery=markevery)    
# Plotting list_2 error bars at marker points
axs[2].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_2[0][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_2[0][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='r', 
)
axs[2].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_2[1][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_2[1][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='m', 
)
axs[2].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_2[2][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_2[2][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='b', 
)
axs[2].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_2[3][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_2[3][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='g', 
)
axs[2].errorbar(range(0, T, markevery), avg_cum_rs_reg_list_2[4][::markevery].flatten(),
                yerr=var_cum_rs_reg_list_2[4][::markevery].flatten()*alpha_error,
                fmt='none', capsize=5, capthick=2, c='darkred', 
)
# Labeling the axes
axs[2].set_xlabel('Round t')
axs[2].set_ylabel('Cumulative RS Regret')
axs[2].set_title('Worst Case Attack')
# axs[2].set_ylim([0, 20])  # Adjust the y-axis limits as needed

# Adding legend and adjusting layout
axs[0].legend()
axs[1].legend()
axs[2].legend()
# Adding legends for tau and epsilon_t
# axs[0].legend(title=r'$\tau=0$, $\epsilon_t=2.5$')
axs[1].legend(title=r'$\tau=0$, $\epsilon_t=2.5$')
# axs[2].legend(title=r'$\tau=0$, $\epsilon_t=2.0$')

plt.tight_layout()
# Save as pdf
dir = 'figures'
os.makedirs(dir, exist_ok=True)
# plt.savefig(os.path.join(dir, 'RS_regret_inv_pend_res8_N_100.pdf'), format='pdf', bbox_inches='tight')
# Display the plot
plt.show()